In [ ]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "07_tfidf_real"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


In [ ]:
# 2. Carregar dados processados do novo pipeline multisource
import pandas as pd, numpy as np

# ATUALIZADO: Usar dados do pipeline multisource (NB 12-14)
ibov_file = os.path.join(PROC_PATH, "ibovespa_clean.csv")
news_file = os.path.join(PROC_PATH, "news_clean.parquet")

# Fallback para arquivo antigo se novo não existir
if not os.path.exists(news_file):
    news_file = os.path.join(PROC_PATH, "noticias_real_clean.csv")
    print(f"⚠️ Usando arquivo legado: {news_file}")
    print("   Recomendado: Execute notebooks 12-14 para gerar news_clean.parquet")

if news_file.endswith('.parquet'):
    df_news = pd.read_parquet(news_file)
else:
    df_news = pd.read_csv(news_file)

df_ibov = pd.read_csv(ibov_file)

# Normalizar datas
if "date" in df_ibov.columns: df_ibov = df_ibov.rename(columns={"date":"data"})
if "published_at" in df_news.columns: df_news = df_news.rename(columns={"published_at":"data"})
elif "date" in df_news.columns: df_news = df_news.rename(columns={"date":"data"})

df_ibov["data"] = pd.to_datetime(df_ibov["data"])

df_news["data"] = pd.to_datetime(df_news["data"])print("Datas NEWS:", df_news["data"].min(), "->", df_news["data"].max())

print("Datas IBOV:", df_ibov["data"].min(), "->", df_ibov["data"].max())

# Garantir coluna clean_textprint("IBOV:", df_ibov.shape, "| NEWS:", df_news.shape)

if "clean_text" not in df_news.columns:

    if "texto" in df_news.columns:        df_news["clean_text"] = ""

        df_news["clean_text"] = df_news["texto"].fillna("")    else:

In [ ]:
# 3. Agregar textos por dia e criar alvo do IBOV
# Agrega todas as notícias do mesmo dia
df_text = df_news.groupby("data", as_index=False)["clean_text"].apply(
    lambda s: " ".join([str(x) for x in s if str(x).strip()])
)

df_ibov = df_ibov.sort_values("data").reset_index(drop=True)
df_ibov["ret1"] = df_ibov["close"].pct_change().shift(-1)
df_ibov["y"] = (df_ibov["ret1"] > 0).astype(int)
df_target = df_ibov.dropna(subset=["ret1"]).copy()

# Merge
df = pd.merge(df_target[["data","close","y"]], df_text, on="data", how="inner").sort_values("data")

# Validação: exigir dados
if df.shape[0] == 0:
    raise ValueError(
        "❌ ERRO: Nenhuma interseção entre IBOV e notícias. "
        "Execute os notebooks 12-14 (pipeline multisource) "
        "ou 05-06 (coleta legada) para gerar dados antes de prosseguir."
    )

print("Após merge:", df.shape)
display(df.head())

In [ ]:
# 4. TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

docs = df["clean_text"].fillna("").replace("", "vazio").tolist()

vectorizer = TfidfVectorizer(
    min_df=1,
    max_df=1.0,
    ngram_range=(1,2),
    strip_accents="unicode",
    lowercase=True,
    max_features=500
)

X = vectorizer.fit_transform(docs)
y = df["y"].astype(int).reset_index(drop=True)

print("Matriz TF-IDF:", X.shape, "| y:", y.shape)

In [ ]:
# 5. Avaliação adaptativa
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

def evaluate_timeaware(X, y, models):
    n = X.shape[0]
    results = {}

    if n < 2:
        print(f"⚠️ Dataset com {n} amostra(s): impossível treinar.")
        return results

    # Ajuste automático de splits
    if n >= 30:
        n_splits = 5
    elif n >= 10:
        n_splits = 3
    elif n >= 5:
        n_splits = 2
    else:
        n_splits = 0

    if n_splits >= 2:
        tscv = TimeSeriesSplit(n_splits=n_splits)
        for name, model in models.items():
            aucs, mdas = [], []
            for tr, te in tscv.split(np.arange(n)):
                model.fit(X[tr], y.iloc[tr])
                proba = model.predict_proba(X[te])[:,1]
                pred  = (proba > 0.5).astype(int)
                aucs.append(roc_auc_score(y.iloc[te], proba))
                mdas.append(accuracy_score(y.iloc[te], pred))
            results[name] = {"AUC": float(np.mean(aucs)), "MDA": float(np.mean(mdas))}
        print(f"✅ Avaliação com TimeSeriesSplit (n_splits={n_splits}, n={n})")
    else:
        print(f"⚠️ Poucas amostras (n={n}). Usando holdout 50/50.")
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.5, shuffle=False
        )
        for name, model in models.items():
            model.fit(X_train, y_train)
            proba = model.predict_proba(X_test)[:,1]
            pred  = (proba > 0.5).astype(int)
            results[name] = {
                "AUC": float(roc_auc_score(y_test, proba)) if len(np.unique(y_test)) > 1 else float("nan"),
                "MDA": float(accuracy_score(y_test, pred))
            }
    return results

In [ ]:
# 6. Modelos e execução
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic": LogisticRegression(max_iter=2000),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42)
}

try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42
    )
except:
    print("ℹ️ XGBoost não disponível, seguindo sem ele.")

results = evaluate_timeaware(X, y, models)
print("\nResultados médios (TF-IDF real/dummy):")
print(results)

In [ ]:
# 7. Salvar resultados em JSON (para o dashboard)
import json, os

nb_name = "07_tfidf_real"

# results já está definido pelos modelos
results_dict = {k: {"AUC": float(v["AUC"]), "MDA": float(v["MDA"])} for k, v in results.items()}

out_file = os.path.join(PROC_PATH, f"results_{nb_name}.json")
with open(out_file, "w") as f:
    json.dump(results_dict, f, indent=4)

print(f"✅ Resultados salvos em {out_file}")

In [ ]:
# Validar período de cobertura
PERIODO = cfg.get_periodo_estudo()
START_EXPECTED = pd.to_datetime(PERIODO["start"])
END_EXPECTED = pd.to_datetime(PERIODO["end"])

if "data" in df.columns:
    min_date = pd.to_datetime(df["data"].min())
    max_date = pd.to_datetime(df["data"].max())
    print(f"\n=== Validação de Período ===")
    print(f"Esperado: {START_EXPECTED.date()} → {END_EXPECTED.date()}")
    print(f"Obtido:   {min_date.date()} → {max_date.date()}")
    
    if min_date > START_EXPECTED + pd.Timedelta(days=30):
        print(f"⚠️ AVISO: Início dos dados ({min_date.date()}) está >30 dias após o esperado.")
    if max_date < END_EXPECTED - pd.Timedelta(days=30):
        print(f"⚠️ AVISO: Fim dos dados ({max_date.date()}) está >30 dias antes do esperado.")
    if min_date <= START_EXPECTED and max_date >= END_EXPECTED:
        print("✅ Período validado com sucesso!")